In [ ]:
# ============================================================
# TP4 - Fetes et consommation au Senegal
# ============================================================
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score, silhouette_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

PLAFOND_JOURS = 60
FENETRE_PROXIMITE = 14
TRAIN_FIN = 2023
VAL_ANNEE = 2024
TEST_ANNEE = 2025
CAT_MOUTON = "betail"


In [ ]:
def normaliser_texte(s):
    """Supprime accents, uniformise separateurs et passe en minuscules."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[\s\-_']+", " ", s).strip().lower()


def slug(nom):
    """Produit un identifiant stable e partir d'un libelle."""
    return normaliser_texte(nom).replace(" ", "_")


def wape(y, y_pred):
    """WAPE = somme des erreurs absolues / somme des ventes reelles."""
    y, y_pred = np.asarray(y, float), np.asarray(y_pred, float)
    denominateur = np.abs(y).sum()
    if not len(y) or denominateur == 0:
        raise ValueError("Le WAPE exige des observations et un total reel non nul.")
    return np.abs(y - y_pred).sum() / denominateur


def biais(y, y_pred):
    """Ratio total prevu / total reel (< 1 : sous-estimation)."""
    y, y_pred = np.asarray(y, float), np.asarray(y_pred, float)
    denominateur = np.asarray(y, float).sum()
    if not len(y) or denominateur == 0:
        raise ValueError("Le biais exige des observations et un total reel non nul.")
    return np.asarray(y_pred, float).sum() / denominateur


def dates_proches_fete(calendrier, rayon=3):
    """Dates situees e +/- rayon jours de chaque debut de fete."""
    plages = [pd.date_range(d - pd.Timedelta(days=rayon),
                            d + pd.Timedelta(days=rayon))
              for d in calendrier["date"].dropna()]
    if not plages:
        return pd.DatetimeIndex([])
    return pd.DatetimeIndex(np.concatenate([p.values for p in plages])).unique()


In [ ]:
DATA_DIR = Path.cwd()
FICHIERS_REQUIS = [
    "ventes_journalieres.csv", "magasins.csv", "enquete_menages.csv",
    "categories.csv", "calendrier_evenements.csv",
]
manquants = [nom for nom in FICHIERS_REQUIS if not (DATA_DIR / nom).is_file()]
if manquants:
    raise FileNotFoundError(
        f"Fichiers CSV introuvables dans {DATA_DIR.resolve()} : {manquants}. "
        "Placez le notebook dans le dossier des donnees ou definissez DATA_DIR."
    )


def lire_csv(nom, **kwargs):
    """Lit un CSV UTF-8 et signale explicitement les erreurs de lecture."""
    return pd.read_csv(DATA_DIR / nom, encoding="utf-8-sig", **kwargs)


def verifier_colonnes(df, attendues, nom):
    absentes = sorted(set(attendues) - set(df.columns))
    if absentes:
        raise ValueError(f"Colonnes absentes de {nom} : {absentes}")


ventes_brut = lire_csv("ventes_journalieres.csv", parse_dates=["date"],
                       low_memory=False)
magasins = lire_csv("magasins.csv", parse_dates=["date_ouverture"])
enquete = lire_csv("enquete_menages.csv", low_memory=False)
categories_ref = lire_csv("categories.csv")
calendrier = lire_csv("calendrier_evenements.csv",
                      parse_dates=["date_debut", "date_fin"])

verifier_colonnes(ventes_brut, ["date", "magasin_id", "categorie", "quantite",
                                "prix_unitaire_fcfa", "promo",
                                "part_paiement_mobile"], "ventes_journalieres.csv")
verifier_colonnes(magasins, ["magasin_id", "ville", "region", "date_ouverture"],
                  "magasins.csv")
verifier_colonnes(categories_ref, ["categorie", "unite", "famille",
                                    "prix_2016_fcfa"], "categories.csv")
verifier_colonnes(calendrier, ["evenement", "date_debut", "date_fin"],
                  "calendrier_evenements.csv")
verifier_colonnes(enquete, ["annee", "menage_id", "evenement", "depense_totale_fcfa",
                            "dep_alimentation_fcfa", "dep_habillement_fcfa",
                            "dep_betail_fcfa", "dep_transport_ziara_fcfa",
                            "dep_cadeaux_dons_fcfa", "revenu_mensuel_fcfa",
                            "taille_menage", "financement_principal",
                            "endette_apres_fete", "region", "milieu", "religion",
                            "confrerie"], "enquete_menages.csv")

# Le calendrier contient des intervalles (notamment pour le Ramadan) ; les
# distances et fenetres utilisent le premier jour de chaque evenement.
calendrier["date"] = calendrier["date_debut"]
calendrier["evenement"] = calendrier["evenement"].map(slug)
FETES = sorted(set(calendrier["evenement"].unique()) - {"ramadan"})

# Rattache les variantes de libelles aux 15 categories de reference.
lookup_categories = {normaliser_texte(nom): nom
                    for nom in categories_ref["categorie"]}
audit = {"lignes_brutes": len(ventes_brut),
         "doublons_exacts": int(ventes_brut.duplicated().sum()),
         "manquants_par_colonne": ventes_brut.isna().sum().to_dict()}
ventes = ventes_brut.drop_duplicates().copy()
ventes["cat_norm"] = ventes["categorie"].map(normaliser_texte)
unknown = sorted(set(ventes["cat_norm"].dropna()) - set(lookup_categories))
if unknown:
    raise ValueError(f"Categories de ventes absentes de categories.csv : {unknown}")
audit["variantes_categorie"] = int(
    ventes["categorie"].nunique() - ventes["cat_norm"].nunique())
ventes["categorie"] = ventes["cat_norm"].map(lookup_categories)
ventes = ventes.drop(columns="cat_norm")

n0 = len(ventes)
ventes = ventes.dropna(subset=["date", "magasin_id", "categorie", "quantite",
                               "prix_unitaire_fcfa"])
audit["lignes_cible_ou_prix_manquants_supprimees"] = n0 - len(ventes)
ventes["quantite"] = pd.to_numeric(ventes["quantite"], errors="raise")
ventes["prix_unitaire_fcfa"] = pd.to_numeric(
    ventes["prix_unitaire_fcfa"], errors="raise")
ventes["part_paiement_mobile"] = pd.to_numeric(
    ventes["part_paiement_mobile"], errors="raise")
if (ventes["prix_unitaire_fcfa"] <= 0).any():
    raise ValueError("Le prix unitaire doit etre strictement positif.")
if ventes["part_paiement_mobile"].isna().any() or not ventes[
        "part_paiement_mobile"].between(0, 1).all():
    raise ValueError("part_paiement_mobile doit etre renseignee entre 0 et 1.")
ventes["promo"] = ventes["promo"].fillna(0).astype(int)

# Les quantites negatives representent des retours et ne mesurent pas la demande.
retours = ventes["quantite"] < 0
audit["retours_exclus"] = int(retours.sum())
ventes = ventes.loc[~retours].copy()

# Z robuste sur log1p, estime hors des jours proches des fetes.
proches = dates_proches_fete(calendrier, rayon=3)
ventes["proche_fete"] = ventes["date"].isin(proches)
ventes["log_q"] = np.log1p(ventes["quantite"])
GRP = ["magasin_id", "categorie"]
stats = (ventes.loc[~ventes["proche_fete"]].groupby(GRP)["log_q"]
         .agg(med="median", mad=lambda s: (s - s.median()).abs().median())
         .reset_index())
ventes = ventes.merge(stats, on=GRP, how="left", validate="many_to_one")
z = 0.6745 * (ventes["log_q"] - ventes["med"]) / ventes["mad"].replace(0, np.nan)
ventes["aberrant"] = (z.abs() > 6) & (~ventes["proche_fete"])
audit["aberrants_hors_fete"] = int(ventes["aberrant"].sum())
ventes = ventes.loc[~ventes["aberrant"]].drop(
    columns=["med", "mad", "log_q", "aberrant", "proche_fete"])

ids_inconnus = sorted(set(ventes["magasin_id"]) - set(magasins["magasin_id"]))
if ids_inconnus:
    raise ValueError(f"Magasins de ventes absents de magasins.csv : {ids_inconnus}")
jours_manquants = magasins.set_index("magasin_id")[["date_ouverture"]].copy()
dernier_jour = ventes.groupby("magasin_id")["date"].max()
jours_manquants["dernier_jour"] = dernier_jour
jours_manquants["jours_observes"] = ventes.groupby("magasin_id")["date"].nunique()
jours_manquants["jours_attendus"] = (
    jours_manquants["dernier_jour"] - jours_manquants["date_ouverture"]
).dt.days + 1
jours_manquants["jours_manquants"] = (
    jours_manquants["jours_attendus"] - jours_manquants["jours_observes"])
print({k: v for k, v in audit.items() if k != "manquants_par_colonne"})
display(jours_manquants)


In [ ]:
CATEGORIES = categories_ref["categorie"].tolist()
PLAGE = pd.date_range(ventes["date"].min(), ventes["date"].max())

ventes_jour = (ventes.groupby(["date", "magasin_id", "categorie"], as_index=False)
               .agg(quantite=("quantite", "sum"), promo=("promo", "max")))

# La grille commence e la date d'ouverture declaree, meme si une premiere vente
# manque dans le fichier ; les absences deviennent alors des zeros de demande.
debut = magasins[["magasin_id", "date_ouverture"]].rename(
    columns={"date_ouverture": "debut"})
grille = pd.MultiIndex.from_product(
    [PLAGE, magasins["magasin_id"], CATEGORIES],
    names=["date", "magasin_id", "categorie"]).to_frame(index=False)
grille = grille.merge(debut, on="magasin_id", validate="many_to_one")
grille = grille[grille["date"] >= grille["debut"]].drop(columns="debut")

demande = grille.merge(ventes_jour, on=["date", "magasin_id", "categorie"],
                       how="left", validate="one_to_one")
demande["quantite"] = demande["quantite"].fillna(0)
demande["promo"] = demande["promo"].fillna(0).astype(int)
demande = demande.merge(magasins[["magasin_id", "ville", "region"]],
                        on="magasin_id", how="left", validate="many_to_one")
print("Lignes de demande :", len(demande))


In [ ]:
def variables_calendrier(dates, calendrier, plafond=PLAFOND_JOURS):
    """
    Pour chaque date unique :
      av_<fete> : jours avant la prochaine occurrence de la fête (plafonné)
      ap_<fete> : jours depuis la précédente occurrence (plafonné)
      ramadan   : 1 si la date est comprise dans l'intervalle debut-fin du fichier
    Implémentation : pd.merge_asof (aucune boucle sur les lignes de ventes).
    """
    base = pd.DataFrame({"date": pd.DatetimeIndex(pd.unique(pd.to_datetime(dates)))})
    base = base.sort_values("date").reset_index(drop=True)

    # Une boucle sur les fêtes (une dizaine), pas sur les lignes
    for nom in FETES:
        occ = (calendrier.loc[calendrier["evenement"] == nom, ["date"]]
               .rename(columns={"date": "j0"}).sort_values("j0"))
        suivant = pd.merge_asof(base, occ, left_on="date", right_on="j0",
                                direction="forward", allow_exact_matches=True)
        precedent = pd.merge_asof(base, occ, left_on="date", right_on="j0",
                                  direction="backward", allow_exact_matches=True)
        base[f"av_{nom}"] = (suivant["j0"] - suivant["date"]).dt.days.fillna(plafond).clip(upper=plafond)
        base[f"ap_{nom}"] = (precedent["date"] - precedent["j0"]).dt.days.fillna(plafond).clip(upper=plafond)

    # Utilise les bornes observees pour respecter les durees de 29 ou 30 jours.
    ramadan = (calendrier.loc[calendrier["evenement"] == "ramadan",
                              ["date_debut", "date_fin"]]
               .rename(columns={"date_debut": "debut", "date_fin": "fin"})
               .sort_values("debut"))
    r = pd.merge_asof(base, ramadan, left_on="date", right_on="debut",
                      direction="backward")
    base["ramadan"] = (r["debut"].notna() & r["date"].le(r["fin"])).astype(int)
    return base.set_index("date")


# Ajout des variables calendaires et de la distance minimale à une fête
COLS_AV = [f"av_{f}" for f in FETES]
COLS_AP = [f"ap_{f}" for f in FETES]


def ajouter_features(df, calendrier):
    feats = variables_calendrier(df["date"], calendrier)
    out = df.merge(feats, left_on="date", right_index=True, how="left")
    out["dow"] = out["date"].dt.day_of_week
    out["mois"] = out["date"].dt.month
    out["jour_annee"] = out["date"].dt.day_of_year
    out["annee"] = out["date"].dt.year
    out["av_min"] = out[COLS_AV].min(axis=1)
    return out


demande = ajouter_features(demande, calendrier)

# Décalage de la Tabaski d'une année sur l'autre
tab = calendrier.loc[calendrier["evenement"] == "tabaski", ["date"]].sort_values("date").copy()
tab["decalage_jours"] = tab["date"].diff().dt.days
display(tab)


In [ ]:
# Fenetres d'achat et de reference relatives e J0 (en jours)
CONFIG_FENETRES = {f: {"fen": (-7, -1), "ref": (-60, -31)} for f in FETES}


def construire_fenetres(calendrier, config):
    """Une ligne par (date, evenement, occurrence J0, phase)."""
    lignes = []
    for evt, phases in config.items():
        dates = calendrier.loc[calendrier["evenement"] == evt, "date"]
        for j0 in dates:
            for phase in ("fen", "ref"):
                a, b = phases[phase]
                jours = pd.date_range(j0 + pd.Timedelta(days=a),
                                      j0 + pd.Timedelta(days=b))
                lignes.append(pd.DataFrame({"date": jours, "evenement": evt,
                                            "j0": j0, "phase": phase}))
    if not lignes:
        raise ValueError("Aucune fenetre d'evenement n'a pu etre construite.")
    return pd.concat(lignes, ignore_index=True)


fen = construire_fenetres(calendrier, CONFIG_FENETRES)

# Vente quotidienne nationale par categorie
ventes_nat = demande.groupby(["date", "categorie"], as_index=False)["quantite"].sum()
panel = fen.merge(ventes_nat, on="date", how="left")

moy = panel.groupby(["evenement", "phase", "categorie"])["quantite"].mean().unstack("phase")
moy["uplift"] = moy["fen"] / moy["ref"].replace(0, np.nan)
top3 = (moy.reset_index().sort_values(["evenement", "uplift"],
                                       ascending=[True, False])
        .groupby("evenement").head(3))
print("Top 3 categories par fete (national) :")
display(top3[["evenement", "categorie", "uplift"]])

compte = fen[fen["phase"] == "fen"].groupby("date")["evenement"].nunique()
print("Dates proches d'au moins 2 fenetres d'achat :", int((compte > 1).sum()))


In [ ]:
# Comparaison Magal / Maouloud par ville (libelles reellement presents)
ventes_ville = demande.groupby(["date", "ville"], as_index=False)["quantite"].sum()
panel_ville = fen.merge(ventes_ville, on="date", how="left")
moy_ville = panel_ville.groupby(["evenement", "phase", "ville"])["quantite"].mean().unstack("phase")
moy_ville["uplift"] = moy_ville["fen"] / moy_ville["ref"].replace(0, np.nan)
villes_cibles = ["Touba", "Tivaouane", "Kaolack", "Dakar Plateau"]
tab_ville = (moy_ville.reset_index()
             .query("evenement in ['magal_de_touba', 'maouloud'] and ville in @villes_cibles")
             .pivot(index="ville", columns="evenement", values="uplift"))
display(tab_ville.round(2))

heat = moy["uplift"].unstack("categorie")
plt.figure(figsize=(14, 5))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdBu_r", center=1)
plt.title("Uplift (fenetre J-7..J-1 / reference J-60..J-31)")
plt.tight_layout()
plt.show()


In [ ]:
# Indicateurs sur dix ans : CA nominal, CA reel, part mobile et prix du mouton
ventes_ca = ventes.assign(
    ca=ventes["quantite"] * ventes["prix_unitaire_fcfa"],
    annee=ventes["date"].dt.year,
    part_mobile=ventes["part_paiement_mobile"].clip(0, 1))
ventes_ca["ca_mobile"] = ventes_ca["ca"] * ventes_ca["part_mobile"]

# Indice de prix de Laspeyres (base 2016), e partir des prix unitaires observes.
prix = ventes_ca.groupby(["annee", "categorie"]).agg(
    ca=("ca", "sum"), q=("quantite", "sum"))
prix["p"] = prix["ca"] / prix["q"].replace(0, np.nan)
p_piv = prix["p"].unstack("categorie")
q_base = prix.xs(2016, level="annee")["q"]
indice = (p_piv.mul(q_base, axis="columns").sum(axis=1)
          / (p_piv.loc[2016] * q_base).sum())

annuel = ventes_ca.groupby("annee").agg(ca_nominal=("ca", "sum"),
                                         ca_mobile=("ca_mobile", "sum"))
annuel["indice_prix"] = indice
annuel["ca_reel_base2016"] = annuel["ca_nominal"] / annuel["indice_prix"]
annuel["part_mobile"] = annuel["ca_mobile"] / annuel["ca_nominal"]

mask_mouton = ventes["categorie"].map(normaliser_texte).str.contains(
    CAT_MOUTON, na=False)
prix_mouton = {}
for j0 in calendrier.loc[calendrier["evenement"] == "tabaski", "date"]:
    m = mask_mouton & ventes["date"].between(
        j0 - pd.Timedelta(days=30), j0 - pd.Timedelta(days=1))
    prix_mouton[j0.year] = ventes.loc[m, "prix_unitaire_fcfa"].mean()
annuel["prix_mouton_avant_tabaski"] = pd.Series(prix_mouton)
display(annuel.round(3))


In [ ]:
def pipeline_ridge(cols_cat, cols_num, alpha):
    """Ridge dans un Pipeline ; cible transformée en log1p puis ramenée par expm1."""
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat),
        ("num", StandardScaler(), cols_num),
    ])
    reg = Pipeline([("prep", prep), ("ridge", Ridge(alpha=alpha))])
    return TransformedTargetRegressor(regressor=reg, func=np.log1p, inverse_func=np.expm1)


def pipeline_hgb(loss, cols_cat, cols_num, **params):
    """HistGradientBoosting avec catégories ordinales natives et perte choisie."""
    prep = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cols_cat),
        ("num", "passthrough", cols_num),
    ])
    hgb = HistGradientBoostingRegressor(
        loss=loss, categorical_features=np.arange(len(cols_cat)),
        random_state=SEED, early_stopping=False, **params)
    return Pipeline([("prep", prep), ("hgb", hgb)])


COLS_CAT = ["magasin_id", "categorie", "dow", "mois"]
COLS_NUM = COLS_AV + COLS_AP + ["ramadan", "promo", "jour_annee"]
HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=50)


def evaluer(df, y_pred):
    """WAPE et biais sur l'année et sur les seules fenêtres de 14 jours avant fête."""
    y = df["quantite"].to_numpy()
    masque = (df["av_min"] <= FENETRE_PROXIMITE).to_numpy()
    return {"WAPE_annee": wape(y, y_pred), "biais_annee": biais(y, y_pred),
            "WAPE_fenetres": wape(y[masque], y_pred[masque]),
            "biais_fenetres": biais(y[masque], y_pred[masque])}


# Séparations strictes
train = demande[demande["annee"] <= TRAIN_FIN]
val = demande[demande["annee"] == VAL_ANNEE]
test = demande[demande["annee"] == TEST_ANNEE]   # ne sera utilisé qu'une fois

# --- Références naïves ---
# (a) Même jour de la semaine, 364 jours plus tôt
decal = demande[["date", "magasin_id", "categorie", "quantite"]].copy()
decal["date"] = decal["date"] + pd.Timedelta(days=364)
naif = demande[["date", "magasin_id", "categorie"]].merge(
    decal.rename(columns={"quantite": "naif364"}), on=["date", "magasin_id", "categorie"], how="left")
demande["naif364"] = naif["naif364"].fillna(0).to_numpy()

# (b) Moyenne de l'année précédente par point de vente, catégorie et jour de la semaine
moy_n1 = (demande.groupby(["annee", "magasin_id", "categorie", "dow"])["quantite"]
          .mean().rename("moy_n1").reset_index())
moy_n1["annee"] += 1  # la moyenne de N-1 sert à prévoir l'année N
demande = demande.merge(moy_n1, on=["annee", "magasin_id", "categorie", "dow"], how="left")
demande["moy_n1"] = demande["moy_n1"].fillna(0)
val = demande[demande["annee"] == VAL_ANNEE]
test = demande[demande["annee"] == TEST_ANNEE]

resultats_val = {
    "naif_364j": evaluer(val, val["naif364"].to_numpy()),
    "moyenne_N-1": evaluer(val, val["moy_n1"].to_numpy()),
}


In [ ]:
# --- Ridge : choix d'alpha sur la validation 2024 ---
resultats_ridge = {}
for alpha in [0.1, 1, 10, 100]:
    m = pipeline_ridge(COLS_CAT, COLS_NUM, alpha)
    m.fit(train[COLS_CAT + COLS_NUM], train["quantite"])
    pred = np.clip(m.predict(val[COLS_CAT + COLS_NUM]), 0, None)
    resultats_ridge[alpha] = evaluer(val, pred)
df_ridge = pd.DataFrame(resultats_ridge).T
best_alpha = df_ridge["WAPE_annee"].idxmin()   # comparaison sur WAPE non arrondi
print("Alpha retenu :", best_alpha)
display(df_ridge.round(4))


In [ ]:
# --- HGB : choix de la perte sur la validation 2024 ---
resultats_hgb = {}
for loss in ["poisson", "squared_error", "absolute_error"]:
    m = pipeline_hgb(loss, COLS_CAT, COLS_NUM, **HGB_PARAMS)
    m.fit(train[COLS_CAT + COLS_NUM], train["quantite"])
    pred = np.clip(m.predict(val[COLS_CAT + COLS_NUM]), 0, None)
    resultats_hgb[loss] = evaluer(val, pred)
df_hgb = pd.DataFrame(resultats_hgb).T
display(df_hgb.round(4))
# Critère : Poisson est une perte de comptage (moyenne conditionnelle positive, bon biais),
# on ne retient une autre perte que si elle améliore nettement le WAPE ET garde un biais proche de 1.
BEST_LOSS = "poisson"


In [ ]:
# --- Réentraînement sur 2016-2024, évaluation UNIQUE sur 2025 ---
train_full = demande[demande["annee"] <= VAL_ANNEE]

ridge_final = pipeline_ridge(COLS_CAT, COLS_NUM, best_alpha).fit(
    train_full[COLS_CAT + COLS_NUM], train_full["quantite"])
hgb_final = pipeline_hgb(BEST_LOSS, COLS_CAT, COLS_NUM, **HGB_PARAMS).fit(
    train_full[COLS_CAT + COLS_NUM], train_full["quantite"])

X_test = test[COLS_CAT + COLS_NUM]
test = test.assign(pred_ridge=np.clip(ridge_final.predict(X_test), 0, None),
                   pred_hgb=np.clip(hgb_final.predict(X_test), 0, None))

resultats_test = pd.DataFrame({
    "naif_364j": evaluer(test, test["naif364"].to_numpy()),
    "moyenne_N-1": evaluer(test, test["moy_n1"].to_numpy()),
    "ridge": evaluer(test, test["pred_ridge"].to_numpy()),
    "hgb_" + BEST_LOSS: evaluer(test, test["pred_hgb"].to_numpy()),
}).T
display(resultats_test.round(4))


In [ ]:
# WAPE par fête sur 2025 (fenêtre des 14 jours avant chaque fête)
lignes = []
for f in FETES:
    m = (test[f"av_{f}"] <= FENETRE_PROXIMITE).to_numpy()
    lignes.append({"fete": f, "jours_lignes": int(m.sum()),
                   "WAPE_hgb": wape(test.loc[m, "quantite"], test.loc[m, "pred_hgb"])})
display(pd.DataFrame(lignes).round(4))


In [ ]:
# --- Les modèles sous-estiment-ils les pics e Trois pics de 2025 ---
jour = test.groupby("date")[["quantite", "pred_hgb"]].sum()
candidats = jour.sort_values("quantite", ascending=False)
pics = []
for d in candidats.index:                          # top ~30 jours, pas d'espace de 7 jours
    if all(abs((d - p).days) >= 7 for p in pics):
        pics.append(d)
    if len(pics) == 3:
        break

rows = []
for d in pics:
    fenetre = jour.loc[d - pd.Timedelta(days=3): d + pd.Timedelta(days=3)]
    rows.append({"pic": d.date(), "reel_7j": fenetre["quantite"].sum(),
                 "prevu_7j": fenetre["pred_hgb"].sum(),
                 "ratio_prevu_reel": fenetre["pred_hgb"].sum() / fenetre["quantite"].sum()})
display(pd.DataFrame(rows).round(3))


In [ ]:
# Une ligne par menage x annee x fete, avec les noms du fichier enquete.
enq = enquete.drop_duplicates().rename(columns={
    "evenement": "fete",
    "depense_totale_fcfa": "depense",
    "endette_apres_fete": "endettement",
    "financement_principal": "mode_financement",
    "revenu_mensuel_fcfa": "revenu_mensuel",
    "milieu": "zone",
}).copy()
enq["fete"] = enq["fete"].map(slug).replace(
    {"korite_(avec_ramadan)": "korite"})
enq["zone"] = enq["zone"].fillna("non renseigne")
enq["religion"] = enq["religion"].fillna("non renseignee")
enq["confrerie"] = enq["confrerie"].fillna("non applicable")

enq_mf = (enq.groupby(["menage_id", "annee", "fete"], as_index=False)
          .agg(depense=("depense", "sum"),
               endettement=("endettement", "max"),
               mode_financement=("mode_financement", "first"),
               revenu_mensuel=("revenu_mensuel", "first"),
               taille_menage=("taille_menage", "first"),
               zone=("zone", "first"), region=("region", "first"),
               religion=("religion", "first"),
               confrerie=("confrerie", "first")))

desc = (enq_mf.groupby(["fete", "mode_financement"])
        .agg(depense_mediane=("depense", "median"),
             taux_endettement=("endettement", "mean"),
             n=("depense", "size"))
        .reset_index())
display(desc.round(3))


In [ ]:
NUM_FEAT = ["revenu_mensuel", "taille_menage"]
CAT_FEAT = ["fete", "mode_financement", "zone", "region", "religion", "confrerie"]
FEATURES = NUM_FEAT + CAT_FEAT
TARGET = "endettement"

# 2016-2022 : apprentissage ; 2023 : choix du modele ; 2024-2025 : test final.
tr = enq_mf[enq_mf["annee"] <= 2022]
va = enq_mf[enq_mf["annee"] == 2023]
tr_final = enq_mf[enq_mf["annee"] <= 2023]
te = enq_mf[enq_mf["annee"] >= 2024]
if tr.empty or va.empty or te.empty:
    raise ValueError("Le decoupage temporel de l'enquete exige des donnees 2016-2025.")


def pipe_logit(features_num, features_cat):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), features_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
    ])
    return Pipeline([("prep", prep),
                     ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])


def pipe_hgb_clf(features_num, features_cat):
    prep = ColumnTransformer([
        ("num", "passthrough", features_num),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                unknown_value=-1), features_cat),
    ])
    clf = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, random_state=SEED,
        categorical_features=np.arange(len(features_num),
                                       len(features_num) + len(features_cat)))
    return Pipeline([("prep", prep), ("clf", clf)])


def metriques_clf(y, p):
    if pd.Series(y).nunique() != 2:
        raise ValueError("Les metriques ROC AUC/PR AUC exigent les deux classes.")
    return {"ROC_AUC": roc_auc_score(y, p),
            "PR_AUC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p)}


log_val = pipe_logit(NUM_FEAT, CAT_FEAT).fit(tr[FEATURES], tr[TARGET])
hgb_val = pipe_hgb_clf(NUM_FEAT, CAT_FEAT).fit(tr[FEATURES], tr[TARGET])
res_val_clf = pd.DataFrame({
    "logistique": metriques_clf(va[TARGET], log_val.predict_proba(va[FEATURES])[:, 1]),
    "hgb": metriques_clf(va[TARGET], hgb_val.predict_proba(va[FEATURES])[:, 1]),
}).T
display(res_val_clf.round(4))
meilleur_nom = res_val_clf["ROC_AUC"].idxmax()

# Reentraenement sans les donnees de test, puis evaluation unique sur 2024-2025.
log_m = pipe_logit(NUM_FEAT, CAT_FEAT).fit(tr_final[FEATURES], tr_final[TARGET])
hgb_m = pipe_hgb_clf(NUM_FEAT, CAT_FEAT).fit(tr_final[FEATURES], tr_final[TARGET])
meilleur = hgb_m if meilleur_nom == "hgb" else log_m
res_clf = pd.DataFrame({
    "logistique": metriques_clf(te[TARGET], log_m.predict_proba(te[FEATURES])[:, 1]),
    "hgb": metriques_clf(te[TARGET], hgb_m.predict_proba(te[FEATURES])[:, 1]),
}).T
display(res_clf.round(4))


In [ ]:
# Importance par permutation (sur le test, le modèle n'est jamais réentraîné ici)
perm = permutation_importance(meilleur, te[FEATURES], te[TARGET], scoring="roc_auc",
                              n_repeats=10, random_state=SEED)
imp = (pd.DataFrame({"variable": FEATURES, "baisse_AUC": perm.importances_mean})
       .sort_values("baisse_AUC", ascending=False))
display(imp.round(4))


In [ ]:
# equite : comparaison avec un modele entraene sans religion ni confrerie.
FEATURES_EQ = [f for f in FEATURES if f not in ["religion", "confrerie"]]
NUM_EQ = NUM_FEAT
CAT_EQ = [c for c in CAT_FEAT if c not in ["religion", "confrerie"]]
modele_eq = (pipe_hgb_clf(NUM_EQ, CAT_EQ) if meilleur_nom == "hgb"
             else pipe_logit(NUM_EQ, CAT_EQ)).fit(tr_final[FEATURES_EQ], tr_final[TARGET])
p_eq = modele_eq.predict_proba(te[FEATURES_EQ])[:, 1]
p_full = meilleur.predict_proba(te[FEATURES])[:, 1]

print("Global, avec religion :", round(roc_auc_score(te[TARGET], p_full), 4))
print("Global, sans religion :", round(roc_auc_score(te[TARGET], p_eq), 4))
te_eq = te.assign(p_eq=p_eq, p_full=p_full)
for g, d in te_eq.groupby("religion"):
    if d[TARGET].nunique() == 2 and len(d) > 50:
        print(f"{g} : AUC avec={roc_auc_score(d[TARGET], d['p_full']):.3f} "
              f"sans={roc_auc_score(d[TARGET], d['p_eq']):.3f} (n={len(d)})")


In [ ]:
# Segmentation 2025 e partir des postes de depense disponibles en colonnes.
POSTES = {
    "alimentation": "dep_alimentation_fcfa",
    "habillement": "dep_habillement_fcfa",
    "betail": "dep_betail_fcfa",
    "transport_ziara": "dep_transport_ziara_fcfa",
    "cadeaux_dons": "dep_cadeaux_dons_fcfa",
}
enq25 = enquete.loc[enquete["annee"] == 2025].copy()
depense_menage = enq25.groupby("menage_id").agg(
    **{nom: (col, "sum") for nom, col in POSTES.items()},
    revenu_mensuel=("revenu_mensuel_fcfa", "first"))
depense_poste = depense_menage[list(POSTES)]
total = depense_poste.sum(axis=1)
parts = depense_poste.div(total.replace(0, np.nan), axis=0)
revenu = depense_menage["revenu_mensuel"]
ok = (total > 0) & (revenu > 0) & revenu.notna()
effort = np.log(total[ok] / revenu[ok])
X_seg = parts.loc[ok].join(effort.rename("effort"))
if len(X_seg) < 6:
    raise ValueError("La segmentation K-means exige au moins six menages valides.")
X_seg = pd.DataFrame(StandardScaler().fit_transform(X_seg),
                     index=X_seg.index, columns=X_seg.columns)

scores = {}
for k in range(2, min(6, len(X_seg) - 1) + 1):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_seg)
    taille_min = np.bincount(km.labels_).min() / len(X_seg)
    if taille_min >= 0.05:
        scores[k] = silhouette_score(
            X_seg, km.labels_, sample_size=min(5000, len(X_seg)),
            random_state=SEED)
if not scores:
    raise ValueError("Aucun k ne respecte la taille minimale de groupe de 5 %.")
print("Silhouette par k :", {k: round(v, 3) for k, v in scores.items()})
K_OPT = max(scores, key=scores.get)

km_final = KMeans(n_clusters=K_OPT, n_init=10, random_state=SEED).fit(X_seg)
profil = (X_seg.assign(groupe=km_final.labels_)
          .groupby("groupe").mean().round(2))
profil["effectif"] = np.bincount(km_final.labels_)
display(profil)


In [ ]:
# La grille 2026 exige des occurrences de fetes en 2026.
if calendrier["date"].dt.year.max() < 2026:
    raise ValueError("Ajoutez les dates 2026 au calendrier avant de prevoir la demande.")

dates_2026 = pd.date_range("2026-01-01", "2026-12-31")
magasins_ref = magasins[["magasin_id", "ville", "region"]].drop_duplicates("magasin_id")
grille_2026 = pd.MultiIndex.from_product(
    [dates_2026, magasins_ref["magasin_id"], CATEGORIES],
    names=["date", "magasin_id", "categorie"]).to_frame(index=False)
grille_2026 = grille_2026.merge(magasins_ref, on="magasin_id",
                                validate="many_to_one")
grille_2026["promo"] = 0
# Hypothese du scenario de base : aucune promotion future n'est connue.
grille_2026 = ajouter_features(grille_2026, calendrier)
print("Lignes 2026 :", len(grille_2026))

modele_2026 = pipeline_hgb(BEST_LOSS, COLS_CAT, COLS_NUM, **HGB_PARAMS).fit(
    demande[COLS_CAT + COLS_NUM], demande["quantite"])
grille_2026["prevision"] = np.clip(
    modele_2026.predict(grille_2026[COLS_CAT + COLS_NUM]), 0, None)


def total_fenetre(df, col, evt, annee, ville=None):
    """Somme sur les sept jours precedant J0 pour un evenement/une annee."""
    j0 = calendrier.loc[(calendrier["evenement"] == evt)
                        & (calendrier["date"].dt.year == annee), "date"]
    if j0.empty:
        return np.nan
    debut_fenetre = j0.iloc[0] - pd.Timedelta(days=7)
    fin_fenetre = j0.iloc[0] - pd.Timedelta(days=1)
    masque = df["date"].between(debut_fenetre, fin_fenetre)
    if ville is not None:
        masque &= df["ville"] == ville
    return df.loc[masque, col].sum()


comparaison = []
for evt, ville in [("tabaski", None), ("magal_de_touba", "Touba"), ("noel", None)]:
    comparaison.append({"fete": evt, "ville": ville or "national",
                        "reel_2025": total_fenetre(demande, "quantite", evt, 2025, ville),
                        "prevision_2026": total_fenetre(
                            grille_2026, "prevision", evt, 2026, ville)})
df_cmp = pd.DataFrame(comparaison)
df_cmp["variation"] = df_cmp["prevision_2026"] / df_cmp["reel_2025"].replace(0, np.nan) - 1
display(df_cmp.round(3))


In [ ]:
def pipeline_quantile(q, cols_cat, cols_num):
    """HGB avec perte quantile : prévoit le quantile q de la demande."""
    prep = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cols_cat),
        ("num", "passthrough", cols_num),
    ])
    hgb = HistGradientBoostingRegressor(
        loss="quantile", quantile=q, categorical_features=np.arange(len(cols_cat)),
        random_state=SEED, early_stopping=False, **HGB_PARAMS)
    return Pipeline([("prep", prep), ("hgb", hgb)])


# Entraînement 2016-2023, couverture mesurée sur 2024
quantiles = {}
for q in [0.1, 0.5, 0.9]:
    quantiles[q] = pipeline_quantile(q, COLS_CAT, COLS_NUM).fit(
        train[COLS_CAT + COLS_NUM], train["quantite"])

lo = np.clip(quantiles[0.1].predict(val[COLS_CAT + COLS_NUM]), 0, None)
hi = np.clip(quantiles[0.9].predict(val[COLS_CAT + COLS_NUM]), 0, None)
lo, hi = np.minimum(lo, hi), np.maximum(lo, hi)            # corrige les croisements
y = val["quantite"].to_numpy()
couverture = np.mean((y >= lo) & (y <= hi))
print(f"Couverture P10-P90 sur 2024 : {couverture:.1%} (cible ~80%)")


In [ ]:
# Les promotions sont concentrees avant les fetes ; ces ecarts restent descriptifs.
val_promo = demande[demande["annee"] <= VAL_ANNEE].copy()
part_promo_proche = val_promo.loc[val_promo["av_min"] <= 14, "promo"].mean()
part_promo_loin = val_promo.loc[val_promo["av_min"] > 14, "promo"].mean()
print(f"Part des lignes en promo : proche fete {part_promo_proche:.1%} | "
      f"loin {part_promo_loin:.1%}")

moy_promo = val_promo.loc[val_promo["promo"] == 1, "quantite"].mean()
moy_sans_promo = val_promo.loc[val_promo["promo"] == 0, "quantite"].mean()
brut = moy_promo / moy_sans_promo if moy_sans_promo else np.nan
print("Ratio brut (promo / sans promo) :", round(brut, 3))

val_promo["strate"] = pd.cut(val_promo["av_min"],
                             bins=[-1, 7, 14, 30, PLAFOND_JOURS],
                             labels=["0-7j", "8-14j", "15-30j", ">30j"])
diff, poids = [], []
for _, d in val_promo.groupby("strate", observed=True):
    if d["promo"].nunique() == 2:
        diff.append(d.loc[d["promo"] == 1, "quantite"].mean()
                    - d.loc[d["promo"] == 0, "quantite"].mean())
        poids.append((d["promo"] == 1).sum())
effet_strate = np.average(diff, weights=poids) if poids and sum(poids) else np.nan
print("ecart descriptif stratifie (unites/jour) :", round(effet_strate, 3))

# Ce scenario compare deux predictions; il n'identifie pas un effet causal.
hgb_promo = pipeline_hgb(BEST_LOSS, COLS_CAT, COLS_NUM, **HGB_PARAMS).fit(
    train_full[COLS_CAT + COLS_NUM], train_full["quantite"])
X_t = test[COLS_CAT + COLS_NUM].reset_index(drop=True)
y1 = hgb_promo.predict(X_t.assign(promo=1))
y0 = hgb_promo.predict(X_t.assign(promo=0))
print("ecart contrefactuel moyen du modele (unites/ligne) :",
      round(np.mean(y1 - y0), 3))
